# 국토부 정류장 위치정보 기반 좌표 매핑

교통카드 수요 데이터의 승차·하차 정류장명에 국토부 정류장 위치정보의 위도·경도를 연결합니다.

In [ ]:
from pathlib import Path
import re
import pandas as pd

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'gtx_a_seoul_bus_outputs' / 'transport_card'
STOPS_FP = Path(r'C:\\Users\\금경훈\\Desktop\\Gachon\\3-2\\UROP\\국토교통부_전국 버스정류장 위치정보_20251031.csv')
DATES = ['20241017', '20251016']

def normalize_name(value):
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', '', str(value).strip())

def find_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f'컬럼을 찾을 수 없습니다: {candidates}')

location = pd.read_csv(STOPS_FP, encoding='cp949')
name_col = find_col(location, ['정류장명', '정류장 명칭', '버스정류장명'])
lat_col = find_col(location, ['위도', '정류장Y위치값', 'Y좌표'])
lon_col = find_col(location, ['경도', '정류장X위치값', 'X좌표'])
location['_name_key'] = location[name_col].map(normalize_name)
location[lat_col] = pd.to_numeric(location[lat_col], errors='coerce')
location[lon_col] = pd.to_numeric(location[lon_col], errors='coerce')
location = location.dropna(subset=[lat_col, lon_col])

# 같은 이름이 여러 좌표를 가지면 이름만으로 확정하지 않음
counts = location.groupby('_name_key')[[lat_col, lon_col]].nunique()
unique_keys = counts[(counts[lat_col] == 1) & (counts[lon_col] == 1)].index
lookup = location[location['_name_key'].isin(unique_keys)].drop_duplicates('_name_key').set_index('_name_key')
print(f'국토부 위치정보: {len(location):,}건 / 고유 정류장명 매칭 후보: {len(lookup):,}개')

for date in DATES:
    input_fp = DATA_DIR / f'gtx_a_transport_card_{date}_raw_with_stop_names.csv'
    output_fp = DATA_DIR / f'gtx_a_transport_card_{date}_raw_with_coords.csv'
    df = pd.read_csv(input_fp, dtype=str, encoding='utf-8-sig').fillna('')

    ride = lookup.reindex(df['승차정류장명'].map(normalize_name))
    goff = lookup.reindex(df['하차정류장명'].map(normalize_name))
    df['승차위도'] = ride[lat_col].to_numpy()
    df['승차경도'] = ride[lon_col].to_numpy()
    df['하차위도'] = goff[lat_col].to_numpy()
    df['하차경도'] = goff[lon_col].to_numpy()

    front = ['query_date', 'query_route_no', '승차정류장ID', '승차정류장명', '승차위도', '승차경도', '하차정류장ID', '하차정류장명', '하차위도', '하차경도']
    df = df[[c for c in front if c in df.columns] + [c for c in df.columns if c not in front]]
    df.to_csv(output_fp, index=False, encoding='utf-8-sig')
    print(f'{date}: {output_fp.name} 저장 ({len(df):,}건) / 승차 {df["승차위도"].notna().mean():.1%} / 하차 {df["하차위도"].notna().mean():.1%}')